In [ ]:
!pip install ultralytics
import os, shutil

In [ ]:
# Define paths to the C2A dataset
base_path = "/kaggle/input/c2a-dataset/C2A_Dataset/new_dataset3"
train_img_dir = os.path.join(base_path, "train", "images")
train_lbl_dir = os.path.join(base_path, "train", "labels")
val_img_dir   = os.path.join(base_path, "val", "images")
val_lbl_dir   = os.path.join(base_path, "val", "labels")

# Create a working directory for the filtered flood-only dataset
work_dir = "/kaggle/working/flood_yolo"
for subdir in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(os.path.join(work_dir, subdir), exist_ok=True)

In [ ]:
# Filter flood images in training set
flood_train_imgs = [f for f in os.listdir(train_img_dir) if f.startswith("flood_image")]
# Filter flood images in validation set
flood_val_imgs = [f for f in os.listdir(val_img_dir) if f.startswith("flood_image")]

print(f"Found {len(flood_train_imgs)} flood training images and {len(flood_val_imgs)} flood validation images.")

# Copy filtered images and labels into the working directory
for img_file in flood_train_imgs:
    # Define source paths
    src_img = os.path.join(train_img_dir, img_file)
    src_lbl = os.path.join(train_lbl_dir, img_file.replace(".png", ".txt"))
    # Define destination paths
    dst_img = os.path.join(work_dir, "images/train", img_file)
    dst_lbl = os.path.join(work_dir, "labels/train", img_file.replace(".png", ".txt"))
    # Copy the files
    shutil.copy(src_img, dst_img)
    shutil.copy(src_lbl, dst_lbl)

for img_file in flood_val_imgs:
    src_img = os.path.join(val_img_dir, img_file)
    src_lbl = os.path.join(val_lbl_dir, img_file.replace(".png", ".txt"))
    dst_img = os.path.join(work_dir, "images/val", img_file)
    dst_lbl = os.path.join(work_dir, "labels/val", img_file.replace(".png", ".txt"))
    shutil.copy(src_img, dst_img)
    shutil.copy(src_lbl, dst_lbl)

In [ ]:
# Create a dataset YAML file for the flood subset
dataset_yaml = f"""
path: {work_dir}
train: images/train
val: images/val
names:
  0: human
"""
with open(os.path.join(work_dir, "flood.yaml"), "w") as f:
    f.write(dataset_yaml)
print("Flood-only dataset YAML created at", os.path.join(work_dir, "flood.yaml"))

In [ ]:
from ultralytics import YOLO
# Load pretrained RT-DETR (official HF version)
model = YOLO("yolo11l.pt")

In [ ]:
results = model.train(
    data=os.path.join(work_dir, "flood.yaml"),  # use the flood-only dataset
    epochs=50,
    batch=4,
    imgsz=960,
    workers=2,
    lr0=0.01,               # Initial learning rate
    lrf=0.01,               # Final learning rate fraction (end of cosine schedule)
    momentum=0.937,         # Momentum for optimizer
    weight_decay=0.0005,    # Helps regularization
    warmup_epochs=3,        # Warmup for stable initial convergence
    augment=True,
    #optimizer="AdamW",
    patience=10       # early stopping if no improvement
    #close_mosaic=10,  # disable mosaic augmentation after 10 epochs
    #degrees=20,       # augmentation: rotation range
    #translate=0.2,    # augmentation: translation
    #scale=0.5,        # augmentation: scale (zoom) range
    #shear=10,         # augmentation: shear angle
    #flipud=0.1,       # augmentation: vertical flip probability
    #hsv_h=0.015,      # augmentation: HSV hue
    #hsv_s=0.7,        # augmentation: HSV saturation
    #hsv_v=0.4         # augmentation: HSV value (brightness)
)

In [ ]:
# Get path from results object
best_model_path = os.path.join(results.save_dir, "weights", "best.pt")

# Copy to a fixed known location
shutil.copy(best_model_path, "/kaggle/working/flood_best.pt")
print("Finetuned model saved to /kaggle/working/flood_best.pt")

In [ ]:
# Load the best model
ft_model = YOLO("/kaggle/working/flood_best.pt")

# Validate on the flood val set again
ft_metrics = ft_model.val(data=os.path.join(work_dir, "flood.yaml"), split="val", imgsz=960, batch=4)

# Extract metrics
ft_precision = ft_metrics.box.mp
ft_recall    = ft_metrics.box.mr
ft_map50     = ft_metrics.box.map50
ft_map5095   = ft_metrics.box.map

print(f"Precision: {ft_precision:.4f}")
print(f"Recall: {ft_recall:.4f}")
print(f"mAP@0.5: {ft_map50:.4f}")
print(f"mAP@0.5:0.95: {ft_map5095:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import random

val_images_dir = os.path.join(work_dir, "images/val")
val_images = os.listdir(val_images_dir)
sample_imgs = random.sample(val_images, 5)  # pick 5 random validation images

for img_file in sample_imgs:
    img_path = os.path.join(val_images_dir, img_file)
    results = ft_model.predict(source=img_path, conf=0.4)  # get predictions (conf=0.4 threshold)
    plotted_img = results[0].plot()  # draw boxes on the image
    
    # Display the image with detections
    plt.figure(figsize=(8, 8))
    plt.imshow(plotted_img)
    plt.title(f"Predictions on {img_file}")
    plt.axis('off')
    plt.show()